## Gateway to Research data

- Fetch companies
- Process them

In [ ]:
import pandas as pd

from src import PROJECT_DIR, logging
from src import utils as src_utils

from discovery_utils.getters import gtr
from discovery_utils.utils import (
    analysis_gtr,
    analysis,
    charts,
    google,
    google_slides,
)

PROJECT_NAME = src_utils.PROJECT_NAME
OUTPUT_DIR = src_utils.OUTPUT_DIR / "mission_radar"

In [ ]:
CONFIG_NAMES = [
    "bioenergy",
    "biomass_heating",
    "built_environment",
    "ccus",
    "district_heating",
    "energy_efficiency",
    "energy_grid", 
    "geothermal_energy",
    "green_skills",
    "heat_pumps",
    "heat_storage",
    "hydrogen_energy",
    "hydrogen_heating",
    "micro_chp",
    "solar_thermal",
    "energy_storage",    
    # "renewables_general",    
    "solar",
    "wind"
    # "decarbonisation_general",    
]

In [ ]:
GTR = gtr.GtrGetter()

enrichment_df = (
    pd.read_csv(src_utils.OUTPUT_DIR / "gtr_labelled_projects.csv")
    .assign(topic_labels = lambda df: df.topic_labels.apply(lambda x: x.split(",")))
    .assign(mission_labels = lambda df: df.mission_labels.apply(lambda x: x.split(",")))
)

In [ ]:
def get_ids_from_config(config_name):
    config = src_utils.get_config_dict(config_name)
    selected_df = src_utils.get_projects_from_config(GTR, enrichment_df, config)

    relevant_check_df = pd.read_json(src_utils.OUTPUT_DIR / f"gtr_llm_check_{config_name}.jsonl", lines=True)
    relevant_checked_df = (
        selected_df
        .merge(relevant_check_df[['id', 'is_relevant']], left_on='id', right_on='id', how='left')
    )
    matching_ids = relevant_checked_df.query("is_relevant == 'yes'").id.tolist()

    return matching_ids

low_carbon_heating_configs = [
    "biomass_heating",    
    "district_heating",
    "geothermal_energy",
    "heat_pumps",
    "heat_storage",
    "hydrogen_heating",
    "micro_chp",
    "solar_thermal",
]
lch_ids = []
for config in low_carbon_heating_configs:
    lch_ids.extend(get_ids_from_config(config))
lch_ids = list(set(lch_ids))

In [ ]:
import datetime
# import datetime data type

def date_to_quarter(date: datetime.datetime) -> str:
    return f"{date.year}-Q{date.quarter}"

def get_present_quarter():
    today = datetime.date.today()
    return f"{today.year}-Q{int((today.month - 1) / 3) + 1}"

In [ ]:
def produce_stats(GTR, matching_ids, category_name, prefix = None):
    """ Produce stats for projects and output charts """

    # Figure variables
    if prefix is None:
        prefix = f"{OUTPUT_DIR}/charts/gtr_{category_name}_"
    _scale = 2

    matchings_projects_df = GTR.projects_enriched.query("id in @matching_ids")

    # Overall investment time series, yearly
    ts_df = (
        analysis_gtr.get_timeseries(matchings_projects_df, period='year', min_year=2014, max_year=2025, description_column="abstractText")
        .assign(amount = lambda df: df.amount / 1_000_000)
    )

    fig = charts.ts_bar(
        ts_df,
        variable='n_projects',
        variable_title="Number of projects",
        category_column="_category",
    )
    fig = charts.configure_plots(fig, chart_title=f"Number of projects for {category_name}")
    chart_filename = f"{prefix}n_projects.png"
    fig.save(chart_filename, scale_factor=_scale)

    fig = charts.ts_bar(
        ts_df,
        variable='amount',
        variable_title="Amount, £ millions",
        category_column="_category",
    )
    fig = charts.configure_plots(fig, chart_title="")
    chart_filename = f"{prefix}amount.png"
    fig.save(chart_filename, scale_factor=_scale)

    growth_magnitude_df = (
        analysis.magnitude_growth(ts_df, year_start=2020, year_end=2024)
        .assign(theme=category_name)
        .reset_index()
        .rename(columns={'index': 'variable'})
    )

    #####
    # Overall investment time series, quarterly
    ts_quarterly_df =  (
        analysis_gtr.get_timeseries(
            matchings_projects_df,
            period='quarter',
            min_year=2023,
            max_year=2025,
            description_column="abstractText"
        )
        .assign(amount = lambda df: df.amount / 1_000_000)
        .assign(quarter = lambda df: df.time_period.apply(date_to_quarter))
        .query("quarter <= @get_present_quarter()")
    )


    fig = charts.ts_bar(
        ts_quarterly_df,
        variable='n_projects',
        variable_title="Number of projects",
        category_column="_category",
        time_column="quarter"            
    )
    fig = charts.configure_plots(fig, chart_title=f"Number of projects for {category_name}")
    chart_filename = f"{prefix}quarterly_n_projects.png"
    fig.save(chart_filename, scale_factor=_scale)
    
    fig = charts.ts_bar(
        ts_quarterly_df,
        variable='amount',
        variable_title="Amount, £ millions",
        category_column="_category",
        time_column="quarter"        
    )
    fig = charts.configure_plots(fig, chart_title="")
    chart_filename = f"{prefix}quarterly_amount.png"
    fig.save(chart_filename, scale_factor=_scale)


    present_quarter = get_present_quarter()
    previous_four_quarters = ts_quarterly_df.query("quarter < @present_quarter").sort_values("quarter").tail(4).quarter.tolist()
    previous_four_quarters_mean_df = (
        ts_quarterly_df
        .query("quarter in @previous_four_quarters")
        .assign(_col = "previous_four_quarters")
        .groupby("_col")
        .agg(
            amount=("amount", "mean"),
            n_projects=("n_projects", "mean"),
        )
        .T
        .reset_index()
        .rename(columns={"index": "variable"})
    )

    present_quarter_df = (
        ts_quarterly_df.query("quarter == @present_quarter")
        # rename index to "present_quarter"
        .assign(_col = "magnitude")
        .groupby("_col")
        .agg(
            amount=("amount", "mean"),
            n_projects=("n_projects", "mean"),
        )
        .T.reset_index().rename(columns={"index": "variable"})
        )

    growth_magnitude_quarterly_df = (
        previous_four_quarters_mean_df
        .merge(present_quarter_df, on="variable", how="left")
        .assign(growth = lambda df: (df.magnitude - df.previous_four_quarters) / df.previous_four_quarters * 100)
        .assign(theme=category_name)
    )

    return (
        ts_df,
        ts_quarterly_df,
        growth_magnitude_df,
        growth_magnitude_quarterly_df,
        matchings_projects_df,
    )

In [ ]:
prefix = None
table_prefix = f"{OUTPUT_DIR}/gtr_"

In [ ]:
all_ts_df = []
all_ts_quarterly_df = []
all_growth_magnitude_df = []
all_growth_magnitude_quarterly_df = []
all_projects_df = []

In [ ]:
for config_name in CONFIG_NAMES:
    logging.info(f"Processing {config_name}")
    config = src_utils.get_config_dict(config_name)
    category_name = config["search_recipe"]["category_name"]
    matching_ids = get_ids_from_config(config_name)

    ts_df, ts_quarterly_df, growth_magnitude_df, growth_magnitude_quarterly_df, projects_df = produce_stats(GTR, matching_ids, category_name, prefix)

    all_ts_df.append(ts_df.assign(theme=category_name))
    all_ts_quarterly_df.append(ts_quarterly_df.assign(theme=category_name))
    all_growth_magnitude_df.append(growth_magnitude_df)
    all_growth_magnitude_quarterly_df.append(growth_magnitude_quarterly_df)
    all_projects_df.append(projects_df.assign(theme=category_name))

In [ ]:
category_name = "Low-carbon heating"
matching_ids = lch_ids

ts_df, ts_quarterly_df, growth_magnitude_df, growth_magnitude_quarterly_df, projects_df = produce_stats(GTR, matching_ids, category_name, prefix)

all_ts_df.append(ts_df.assign(theme=category_name))
all_ts_quarterly_df.append(ts_quarterly_df.assign(theme=category_name))
all_growth_magnitude_df.append(growth_magnitude_df)
all_growth_magnitude_quarterly_df.append(growth_magnitude_quarterly_df)
all_projects_df.append(projects_df.assign(theme=category_name))

In [ ]:
all_ts_df = pd.concat(all_ts_df, ignore_index=True)
all_ts_df.to_csv(f"{table_prefix}all_ts_df.csv", index=False)

all_ts_quarterly_df = pd.concat(all_ts_quarterly_df, ignore_index=True)
all_ts_quarterly_df.to_csv(f"{table_prefix}all_ts_quarterly_df.csv", index=False)

all_growth_magnitude_df = pd.concat(all_growth_magnitude_df, ignore_index=True)
all_growth_magnitude_df.to_csv(f"{table_prefix}all_growth_magnitude_df.csv", index=False)

all_growth_magnitude_quarterly_df = pd.concat(all_growth_magnitude_quarterly_df, ignore_index=True)
all_growth_magnitude_quarterly_df.to_csv(f"{table_prefix}all_growth_magnitude_quarterly_df.csv", index=False)

all_projects_df = pd.concat(all_projects_df, ignore_index=True)
all_projects_df.to_csv(f"{table_prefix}all_projects_df.csv", index=False)

In [ ]:
all_growth_magnitude_df

## Baseline stats

In [ ]:
matching_ids = GTR.projects_enriched.id.to_list()
prefix=None
category_name = "All projects"

In [ ]:
ts_df, ts_quarterly_df, growth_magnitude_df, growth_magnitude_quarterly_df, projects_df = produce_stats(GTR, matching_ids, category_name, prefix)

In [ ]:
growth_magnitude_df

In [ ]:
growth_magnitude_quarterly_df